In [ ]:
import argparse, os, re, time, json, threading, csv
import torch
import torch.nn.functional as F
from pynvml import *


In [ ]:
PATTERN = re.compile(r"id38_dc_0_step(\d+)_input\.pt$")

In [1]:
def find_input_files(input_dir):
    files = []
    for fn in os.listdir(input_dir):
        m = PATTERN.match(fn)
        if m:
            n = int(m.group(1))
            files.append((n, os.path.join(input_dir, fn)))
    files.sort(key=lambda x: x[0])
    return [p for _, p in files]

In [2]:
def same_padding(kernel_size, dilation=1, stride=1):
    # aceita int ou tuple
    def pad_1d(k):
        return ((k - 1) * dilation) // 2
    if isinstance(kernel_size, int):
        return pad_1d(kernel_size)
    else:
        return tuple(pad_1d(k) for k in kernel_size)

### NVML Sampling

In [ ]:
class GPUPowerSampler:
    def __init__(self, gpu_index=0, interval=0.2):
        self.gpu_index = gpu_index
        self.interval = interval
        self.samples = []  # (t_rel_s, power_W)
        self._stop = threading.Event()
        self._thread = None
        self._t0 = None

    def start(self):
        nvmlInit()
        self.handle = nvmlDeviceGetHandleByIndex(self.gpu_index)
        self._t0 = time.time()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def _run(self):
        t_prev = time.time()
        while not self._stop.is_set():
            p_mw = nvmlDeviceGetPowerUsage(self.handle)  # mW
            now = time.time()
            self.samples.append((now - self._t0, p_mw / 1000.0))
            sleep_left = self.interval - (now - t_prev)
            t_prev = now
            if sleep_left > 0:
                time.sleep(sleep_left)

    def stop(self):
        # colete mais uma amostra final
        p_mw = nvmlDeviceGetPowerUsage(self.handle)
        now = time.time()
        self.samples.append((now - self._t0, p_mw / 1000.0))
        self._stop.set()
        if self._thread:
            self._thread.join()
        nvmlShutdown()

    def energy_joules(self):
        # Integra trapézios: sum((P_i + P_{i+1})/2 * dt)
        if len(self.samples) < 2:
            return 0.0
        e = 0.0
        for i in range(len(self.samples) - 1):
            t0, p0 = self.samples[i]
            t1, p1 = self.samples[i + 1]
            dt = t1 - t0
            e += (p0 + p1) * 0.5 * dt
        return e


### Convolution

In [3]:
def run_convolution(input_dir, weights_path, bias_path, device="cuda",
                    stride=1, dilation=1, padding_mode="same", dtype="float16",
                    keep_outputs=False, out_dir=None):
    device = torch.device(device)
    torch_dtype = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}[dtype]

    # Carrega pesos/bias
    W = torch.load(weights_path, map_location="cpu")
    b = torch.load(bias_path, map_location="cpu")
    if not isinstance(W, torch.Tensor) or not isinstance(b, torch.Tensor):
        raise ValueError("weights.pt e bias.pt devem conter tensores do PyTorch")

    W = W.to(device=device, dtype=torch_dtype, non_blocking=True)
    b = b.to(device=device, dtype=torch_dtype, non_blocking=True)

    # Detecta dimensionalidade: (outC, inC, K) / (outC, inC, KH, KW) / (outC, inC, KD, KH, KW)
    dims = W.dim()
    if dims == 3:
        conv_type = "1d"
        kernel_size = W.shape[-1]
    elif dims == 4:
        conv_type = "2d"
        kernel_size = (W.shape[-2], W.shape[-1])
    elif dims == 5:
        conv_type = "3d"
        kernel_size = (W.shape[-3], W.shape[-2], W.shape[-1])
    else:
        raise ValueError(f"Formato dos pesos não suportado: shape={tuple(W.shape)}")

    if padding_mode == "same":
        pad = same_padding(kernel_size, dilation=dilation, stride=stride)
    else:
        pad = 0

    files = find_input_files(input_dir)
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado no padrão em {input_dir}")

    outputs_meta = []
    for path in files:
        x = torch.load(path, map_location="cpu")
        if not isinstance(x, torch.Tensor):
            raise ValueError(f"{path} não contém um tensor PyTorch")

        # Normaliza formato: adiciona batch se faltar
        # Esperado: (N, C, L/H,W/D?) conforme conv
        if conv_type == "1d":
            # Aceita (C, L) ou (N, C, L)
            if x.dim() == 2:
                x = x.unsqueeze(0)
            elif x.dim() != 3:
                raise ValueError(f"Entrada 1D deve ser (N,C,L) ou (C,L), mas veio {tuple(x.shape)}")
        elif conv_type == "2d":
            if x.dim() == 3:
                x = x.unsqueeze(0)
            elif x.dim() != 4:
                raise ValueError(f"Entrada 2D deve ser (N,C,H,W) ou (C,H,W), mas veio {tuple(x.shape)}")
        else:  # 3d
            if x.dim() == 4:
                x = x.unsqueeze(0)
            elif x.dim() != 5:
                raise ValueError(f"Entrada 3D deve ser (N,C,D,H,W) ou (C,D,H,W), mas veio {tuple(x.shape)}")

        x = x.to(device=device, dtype=torch_dtype, non_blocking=True)

        # Confere canais
        inC = W.shape[1]
        if x.shape[1] != inC:
            raise ValueError(f"Canais não batem: input C={x.shape[1]} vs pesos inC={inC} em {os.path.basename(path)}")

        # Convolução
        if conv_type == "1d":
            y = F.conv1d(x, W, b, stride=stride, padding=pad, dilation=dilation)
        elif conv_type == "2d":
            y = F.conv2d(x, W, b, stride=stride, padding=pad, dilation=dilation)
        else:
            y = F.conv3d(x, W, b, stride=stride, padding=pad, dilation=dilation)

        # força execução
        torch.cuda.synchronize(device=device)

        if keep_outputs:
            if out_dir is None:
                out_dir = os.path.join(input_dir, "conv_outputs")
            os.makedirs(out_dir, exist_ok=True)
            base = os.path.basename(path).replace("_input.pt", "_conv.pt")
            torch.save(y.detach().cpu(), os.path.join(out_dir, base))

        outputs_meta.append({"input": os.path.basename(path), "output_shape": tuple(y.shape)})

        # libera
        del x, y
        torch.cuda.empty_cache()

    return outputs_meta, conv_type, kernel_size, pad, dtype